## **RunnableConfig**

**RunnableConfig** is the runtime configuration dict passed to every Runnable method (`invoke`, `astream`, etc.), **enabling dependency injection, tracing, retries, and custom fields** like `session_id` without altering inputs.

Think of it as the **metadata "backpack"** that accompanies your input data as it travels through your chain.

While your **input data (the prompt) changes at every step**, the **RunnableConfig stays constant** (or evolves) throughout the entire chain execution. 

It **tells the infrastructure "how" to run the code**, rather than "what" data to process.

In production, you use RunnableConfig to control the environment. It is a dictionary with these critical keys:

- **configurable:** Holds user-specific overrides (like changing models or temperatures).
- **tags:** Lists of strings for filtering in LangSmith (e.g., ["production", "user-id-123"]).
- **callbacks:** Custom functions that trigger on specific events (e.g., logging every token to a custom dashboard).
- **run_name:** A human-readable name that appears in your LangSmith traces.


```
# Full schema
{
    "configurable": {},           # * Custom fields (e.g., {"session_id": "user123"})
    "tags": ["prod", "v1"],       # * LangSmith tags for filtering traces
    "run_name": "qa-chain",       # * Trace name override
    "run_id": "uuid",             # Custom trace ID
    "max_concurrency": 5,         # Batch/stream limits
    "metadata": {"user_id": 123}, # Arbitrary trace metadata
    "callbacks": [handler1],      # List of BaseCallbackHandler (tracing, logging)
    "retry_on": ExceptionType,    # Retry policy
    "timeout": 30.0,              # Seconds
}
```

### **Tagging and Observability**

In [ ]:
# The chain definition remains the same
chain = prompt | model | output_parser

# Production Call: Inject metadata for LangSmith
response = chain.invoke(
    {"question": "How do I fix this bug?"},
    config={
        "run_name": "support_ticket_processor",
        "tags": ["prod", "user_group_A"],
        "metadata": {"ticket_id": "TKT-9982"}
    }
)

**Note: If a user reports a bug, you can search LangSmith for metadata.ticket_id == "TKT-9982" and see the exact trace of what the agent did for that specific request**

### **Runtime Environment Injection with configurable_fields()**

https://reference.langchain.com/python/langchain-core/runnables/base/RunnableSerializable/configurable_fields

Production systems often handle multiple tenants or contexts. Instead of passing tenant_id as a variable in every single prompt, pass it via configurable.

In [ ]:
from langchain_core.runnables import ConfigurableField

# 1. Define a configurable parameter
model = ChatOpenAI(temperature=1).configurable_fields(
    temperature=ConfigurableField(id="runtime_temperature")
)

# 2. Invoke model with default temperature value of 1
response = model.invoke("Analyze this text.")

# 3. Invoke with runtime settings
# You are injecting 'temperature' into the model 
# without changing the chain object at all.
response = model.invoke(
    "Analyze this text.",
    config={
        "configurable": {
            "runtime_temperature": 0.2  # Precision mode for extraction
        }
    }
)

### **Model Switching during Runtime using configurable_alternatives()**

https://reference.langchain.com/python/langchain-core/runnables/base/RunnableSerializable/configurable_alternatives

You write the logic once, and the behavior changes dynamically based on the external config passed in the RunnableConfig.
- Free user? API passes {"configurable": {"model": "gpt-4o-mini"}}.
- Enterprise user? API passes {"configurable": {"model": "gpt-4o"}}.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

# 1. Define your base model
# We mark 'model' as configurable so we can swap it later
model = ChatOpenAI(model="gpt-4o").configurable_alternatives(
    # The field we want to swap
    "model",
    # A unique ID for this configuration
    default_key="openai",
    # Alternative 1: Anthropic
    anthropic=ChatAnthropic(model="claude-3-5-sonnet-20240620"),
    # Alternative 2: Another OpenAI model
    openai_mini=ChatOpenAI(model="gpt-4o-mini")
)

# 2. Build your chain
chain = ChatPromptTemplate.from_template("Tell me a joke about {topic}") | model

# 3. Usage at Runtime
# Switch to Anthropic
response_anthropic = chain.invoke(
    {"topic": "cats"}, 
    config={"configurable": {"model": "anthropic"}}
)

# Switch to OpenAI Mini
response_mini = chain.invoke(
    {"topic": "dogs"}, 
    config={"configurable": {"model": "openai_mini"}}
)

#### **Another Example**
https://reference.langchain.com/python/langchain-core/runnables/configurable/RunnableConfigurableAlternatives

In [1]:
from langchain_core.runnables import ConfigurableField
from langchain_openai import ChatOpenAI

# This creates a RunnableConfigurableAlternatives for Prompt Runnable
# with two alternatives.
prompt = PromptTemplate.from_template(
    "Tell me a joke about {topic}"
).configurable_alternatives(
    ConfigurableField(id="prompt"),
    default_key="joke",
    poem=PromptTemplate.from_template("Write a short poem about {topic}"),
)

# When invoking the created RunnableSequence, you can pass in the
# value for your ConfigurableField's id which in this case will either be
# `joke` or `poem`.
chain = prompt | ChatOpenAI(model="gpt-5.4-mini")

# The `with_config` method brings in the desired Prompt Runnable in your
# Runnable Sequence.
chain.with_config(configurable={"prompt": "poem"}).invoke({"topic": "bears"})

NameError: name 'PromptTemplate' is not defined

In [ ]:
from langchain_core.runnables import ConfigurableField
from langchain_core.runnables.configurable import (
    RunnableConfigurableAlternatives,
)
from langchain_openai import ChatOpenAI

prompt = RunnableConfigurableAlternatives(
    which=ConfigurableField(id="prompt"),
    default=PromptTemplate.from_template("Tell me a joke about {topic}"),
    default_key="joke",
    prefix_keys=False,
    alternatives={
        "poem": PromptTemplate.from_template("Write a short poem about {topic}")
    },
)
chain = prompt | ChatOpenAI(model="gpt-5.4-mini")
chain.with_config(configurable={"prompt": "poem"}).invoke({"topic": "bears"})

In [25]:
from langchain_core.runnables import RunnableConfig

def sum_method(x: int, config: RunnableConfig) -> int:
    print(config["configurable"]) 
    print(config.get("tags", []))
    return x + x

def multiply_method(x: int, config: RunnableConfig) -> int:
    return x * x

In [26]:
# Converting methods to Runnables
from langchain_core.runnables import RunnableLambda

runnable_1 = RunnableLambda(lambda x : sum_method(x))
runnable_2 = RunnableLambda(lambda x : multiply_method(x))

In [28]:
chain = runnable_1 | runnable_2

config = {
    "run_name":"my-llm",
    "tags":["core"],
    "configurable":{"temp": 0.7, "session_id":"123"}
}

chain.invoke(5, config=config) # Permanent config

{'temp': 0.7, 'session_id': '123'}


100